In [1]:
def raster_to_points(input_raster):
    """
    양의 값을 가지는 픽셀을 좌표(Point)로 변환합니다.

    이 함수는 단일 밴드 GeoTIFF 레스터 파일을 열어,
    값이 0보다 큰 픽셀의 중심 좌표와 해당 값을 추출합니다.

    매개변수:
    ----------
    input_raster : str  
        입력할 GeoTIFF(.tif) 파일의 경로입니다.

    반환값:
    ----------
    list of tuple  
        (x, y, value) 형태의 튜플로 구성된 리스트입니다.

    필요 모듈:
    ----------
    import rasterio as rio  
    import numpy as np
    """
    with rio.open(input_raster) as src:
        data = src.read(1)
        transform = src.transform
        points = []
        for row in range(data.shape[0]):
            for col in range(data.shape[1]):
                value = data[row, col]
                if value > 0:
                    x, y = transform * (col + 0.5, row + 0.5)
                    points.append((x, y, value))
    return points


def points_to_gdf(points, crs):
    """
    포인트 리스트를 GeoDataFrame으로 변환합니다.

    각 포인트는 (x, y, value) 형식의 튜플이며,
    이를 기반으로 좌표와 값을 가진 GeoDataFrame을 생성합니다.

    매개변수:
    ----------
    points : list of tuple  
        (x, y, value) 형태의 데이터 리스트입니다.

    crs : int 또는 str  
        좌표계 정보 (예: EPSG 코드).

    반환값:
    ----------
    gpd.GeoDataFrame  
        'value' 열과 포인트 형식의 geometry 열을 포함하는 GeoDataFrame입니다.

    필요 모듈:
    ----------
    from shapely.geometry import Point  
    import geopandas as gpd
    """
    geometries = [Point(x, y) for x, y, _ in points]
    values = [v for _, _, v in points]
    gdf = gpd.GeoDataFrame({'value': values}, geometry=geometries, crs=crs)
    return gdf


def spatial_join_grid(points_gdf, grid_gdf):
    """
    포인트 데이터를 격자와 공간 결합하고, 각 셀별 값을 집계합니다.

    포인트 GeoDataFrame과 격자(폴리곤) GeoDataFrame 간의 spatial join을 수행하여,
    각 격자 셀 내 포인트들의 value 값을 합산합니다.

    매개변수:
    ----------
    points_gdf : gpd.GeoDataFrame  
        포인트 데이터를 포함한 GeoDataFrame입니다.

    grid_gdf : gpd.GeoDataFrame  
        격자 데이터를 포함한 GeoDataFrame입니다.

    반환값:
    ----------
    gpd.GeoDataFrame  
        각 셀마다 'value' 합계가 포함된 격자 GeoDataFrame. (값이 없는 셀은 제거됨)

    필요 모듈:
    ----------
    import geopandas as gpd
    """
    joined = gpd.sjoin(points_gdf, grid_gdf, how="inner", predicate="within")
    grouped = joined.groupby("index_right")["value"].sum()
    grid_gdf["value"] = grouped
    return grid_gdf.dropna(subset=["value"])
